# Trabalho Final - Predição de Diabetes (Pima Indians)

**Disciplina de Inteligência Artificial , Professor Munif , Unicesumar 2026**

## Integrantes
- Pedro Henrique da Silva - RA: 23021607-2
- Victor Hugo Rodrigues de Oliveira - RA: 23418156-2
- Victor Hungo Silva Garcia - RA: 23030968-2

## Contextualização
O diabetes mellitus é uma doença crônica que afeta milhões de pessoas no mundo. A detecção precoce é fundamental para reduzir complicações. Este notebook aplica **KNN** (Parte 1) e **SVM** (Parte 2) para prever diabetes com o dataset **Pima Indians**.

> **Como executar:** menu *Runtime → Run all* (ou Ctrl+F9)

In [ ]:
import os
import sys
from pathlib import Path

REPO_URL = "https://github.com/PedroSilvazDev/trabalho-final-ia-diabetes-colab.git"
REPO_DIR = "trabalho-final-ia-diabetes-colab"

if Path("src").exists():
    root = Path.cwd()
elif Path(REPO_DIR, "src").exists():
    os.chdir(REPO_DIR)
    root = Path.cwd()
else:
    !git clone {REPO_URL}
    os.chdir(REPO_DIR)
    root = Path.cwd()

sys.path.insert(0, str(root))
!pip install -q -r requirements.txt

print("Diretorio:", root)

## 1. Carregar dataset

| Item | Descrição |
|------|-----------|
| Nome | Pima Indians Diabetes Database |
| Origem | Kaggle / UCI ML Repository |
| Registros | 768 amostras |
| Variável alvo | Outcome (0 = sem diabetes, 1 = com diabetes) |

In [ ]:
import pandas as pd
from IPython.display import display

from src.data_loader import load_dataset

df = load_dataset()
print(f"Registros: {len(df)} | Atributos: {df.shape[1] - 1}")
print("\nDistribuicao da variavel alvo:")
print(df["Outcome"].value_counts())
display(df.head())

## 2. Preparação dos dados

- Substituição de zeros inválidos por valores ausentes
- Imputação pela mediana
- Padronização (StandardScaler)
- Divisão estratificada 80% treino / 20% teste

In [ ]:
from src.preprocess import prepare_data

prepared = prepare_data(df)
print(f"Treino: {len(prepared.y_train)} amostras")
print(f"Teste: {len(prepared.y_test)} amostras")

## 3. Treinamento - KNN (Parte 1)

In [ ]:
from src.train import train_knn, print_result_summary

knn_result = train_knn(prepared)
print_result_summary(knn_result, prepared)

## 4. Treinamento - SVM (Parte 2)

In [ ]:
from src.train import train_svm

svm_result = train_svm(prepared)
print_result_summary(svm_result, prepared)

## 5. Avaliação e comparação

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from src.evaluate import (
    plot_confusion_matrix,
    plot_knn_k_search,
    plot_metrics_comparison,
    plot_roc_curves,
)

%matplotlib inline

metrics_df = pd.DataFrame([
    {"modelo": "KNN", **knn_result.metrics},
    {"modelo": "SVM", **svm_result.metrics},
])

display(metrics_df)

In [ ]:
plot_confusion_matrix(prepared.y_test, knn_result.y_pred, "KNN")
plot_confusion_matrix(prepared.y_test, svm_result.y_pred, "SVM")
plot_knn_k_search(knn_result.grid_search)
plot_metrics_comparison(metrics_df)
plot_roc_curves([
    {
        "model_name": "KNN",
        "y_true": prepared.y_test,
        "y_proba": knn_result.y_proba,
        "metrics": knn_result.metrics,
    },
    {
        "model_name": "SVM",
        "y_true": prepared.y_test,
        "y_proba": svm_result.y_proba,
        "metrics": svm_result.metrics,
    },
])

## 6. Comparação dos resultados

O modelo com melhor desempenho geral foi o **KNN**, considerando F1-score e acurácia. O SVM apresentou ROC-AUC ligeiramente superior, indicando melhor capacidade de separar as classes.

- **KNN:** melhor k = 13
- **SVM:** kernel linear, C = 1, gamma = scale

## Conclusão

O projeto demonstra o fluxo completo de uma solução de IA: definição do problema, preparação dos dados, treinamento, avaliação e comparação entre modelos. Ambos os algoritmos classificam a presença de diabetes com desempenho razoável, com leve vantagem do KNN nas métricas principais.